### EVP WEEKLY UPDATE REORT
#### AUTOMATION
Author: Flavia Davila

In [5]:
import pandas as pd

### 1. LOAD DATA

In [6]:
df = pd.read_excel(r"data.xlsx")

In [7]:
df.describe()

,#AÑO,#MES,#DIA,UNIDADES,VENTA NETA,TRANSACCIONES,CLIENTES,PROFIT
count,3867.000000,3867.000000,3867.000000,3867.000000,3.867000e+03,3867.000000,3867.000000,3.867000e+03
mean,2025.163176,6.503491,15.642617,45025.028185,5.212744e+05,6394.005172,3373.297130,1.545582e+05
std,0.369574,3.420545,8.826635,120528.732752,1.375439e+06,16238.206439,8658.639927,4.174077e+05
min,2025.000000,1.000000,1.000000,-36.010000,-1.721410e+03,1.000000,1.000000,-5.738231e+03
25%,2025.000000,4.000000,8.000000,98.600000,1.703045e+03,15.000000,2.000000,3.873799e+02
50%,2025.000000,7.000000,16.000000,4706.000000,4.540247e+04,380.000000,23.000000,1.401028e+04
75%,2025.000000,9.000000,23.000000,8366.000000,1.047794e+05,1034.500000,463.000000,2.909594e+04
max,2026.000000,12.000000,31.000000,559379.000000,6.434510e+06,69550.000000,37538.000000,1.988647e+06


### 2. TRANSFORM DATA

In [9]:
# creating a new field for %margen
df['MARGEN'] = df['PROFIT']/df['VENTA NETA']

In [22]:
df['GRUPO_CANAL'] = (
    df['CANAL_VENTA']
    .isin(['AGREGADORES', 'CONTACT CENTER', 'ECOMMERCE'])
    .map({True: 'DINEX', False: 'OTROS'})
)

In [23]:
df.columns

Index(['UNE', 'CANAL_VENTA', '#AÑO', '#MES', '#DIA', 'UNIDADES', 'VENTA NETA',
       'TRANSACCIONES', 'CLIENTES', 'PROFIT', 'MARGEN', 'GRUPO_CANAL'],
      dtype='object')

In [24]:
df_unes = (
    df
    .groupby(['UNE', '#AÑO', '#MES', '#DIA'], as_index=False)
    .agg({
        'UNIDADES': 'sum',
        'VENTA NETA': 'sum',
        'TRANSACCIONES': 'sum',
        'CLIENTES': 'sum',
        'PROFIT': 'sum',
        'MARGEN': 'mean'  
    })
)


In [26]:
df_canal = (
    df
    .groupby(['GRUPO_CANAL', '#AÑO', '#MES', '#DIA'], as_index=False)
    .agg({
        'UNIDADES': 'sum',
        'VENTA NETA': 'sum',
        'TRANSACCIONES': 'sum',
        'CLIENTES': 'sum',
        'PROFIT': 'sum',
        'MARGEN': 'mean'  
    })
)

In [27]:
df_canal

,GRUPO_CANAL,#AÑO,#MES,#DIA,UNIDADES,VENTA NETA,TRANSACCIONES,CLIENTES,PROFIT,MARGEN
0,DINEX,2025,3,1,27734.04,299059.72,1882,913,7.832836e+04,0.260570
1,DINEX,2025,3,2,24412.85,219162.77,1878,874,5.788882e+04,0.234706
2,DINEX,2025,3,3,18091.44,200108.67,1751,780,5.246788e+04,0.255151
3,DINEX,2025,3,4,17121.10,175919.54,1553,731,4.881049e+04,0.259970
4,DINEX,2025,3,5,18574.26,201146.21,1441,760,5.580567e+04,0.274969
...,...,...,...,...,...,...,...,...,...,...
731,OTROS,2026,2,27,505158.92,6067514.58,74199,40076,1.529390e+06,0.247845
732,OTROS,2026,2,28,486037.88,5755904.36,70926,38831,1.423209e+06,0.245140
733,OTROS,2026,3,1,456262.38,5155264.92,69357,36889,1.348344e+06,0.254365
734,OTROS,2026,3,2,548991.71,6485949.68,77497,41947,1.660032e+06,0.248192


### 3. MTD vs LY & LM COMPARATIVE TABLES

In [ ]:
import datetime
import numpy as np

# 1. Parámetros de fecha
current_date = datetime.datetime.now()
current_year, current_month, current_day = current_date.year, current_date.month, current_date.day

last_month = current_month - 1 if current_month > 1 else 12
year_last_month = current_year if current_month > 1 else current_year - 1

# 2. Filtrado de periodos
df_mtd_2026 = df[(df["#AÑO"] == current_year) & (df["#MES"] == current_month) & (df["#DIA"] <= current_day)].copy()
df_mtd_2025 = df[(df["#AÑO"] == current_year - 1) & (df["#MES"] == current_month) & (df["#DIA"] <= current_day)].copy()
df_mtd_lm = df[(df["#AÑO"] == year_last_month) & (df["#MES"] == last_month) & (df["#DIA"] <= current_day)].copy()

print(f"Análisis al: {current_date.strftime('%Y-%m-%d %H:%M')}")

In [ ]:
# 3. Agregación por UNE
metrics = ["VENTA NETA", "UNIDADES", "CLIENTES", "TRANSACCIONES", "PROFIT"]

agg_2026 = df_mtd_2026.groupby("UNE")[metrics].sum()
agg_2025 = df_mtd_2025.groupby("UNE")[metrics].sum()
agg_lm = df_mtd_lm.groupby("UNE")[metrics].sum()

# 4. Cálculo de Variaciones %
var_vs_ly = ((agg_2026 / agg_2025) - 1)
var_vs_lm = ((agg_2026 / agg_lm) - 1)

# Cálculo de Margen Bruto (Profit / Venta Neta) y su variación
margen_2026 = agg_2026["PROFIT"] / agg_2026["VENTA NETA"]
margen_2025 = agg_2025["PROFIT"] / agg_2025["VENTA NETA"]
margen_lm = agg_lm["PROFIT"] / agg_lm["VENTA NETA"]

var_vs_ly["MARGEN BRUTO"] = margen_2026 - margen_2025 # En puntos porcentuales como el screenshot
var_vs_lm["MARGEN BRUTO"] = margen_2026 - margen_lm

# 5. Formatear como la tabla del screenshot (Métricas en filas, UNE en columnas)
table_ly = var_vs_ly.T
table_lm = var_vs_lm.T

print("\n--- VARIACIÓN % vs AÑO ANTERIOR (LY) ---")
display(table_ly.style.format("{:.2%}"))

print("\n--- VARIACIÓN % vs MES ANTERIOR (LM) ---")
display(table_lm.style.format("{:.2%}"))